# 01 Data audit

Before any modelling we check that the data is what we think it is.

What we check here:

1. Files load and the shapes make sense
2. Missing values, and the literal "NA" category
3. Duplicates
4. Date coverage
5. Test customers versus train customers
6. Category values versus the label config
7. Negative values, returns and adjustment rows
8. We rebuild the official scaling constants to prove our label code matches the organisers

Run this from the `notebooks` folder with the raw files in `data/01_raw`.

In [ ]:
# setup
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

RAW = Path("../data/01_raw")

## 1. Load the files

In [ ]:
# read ID and product codes as text, and keep the category "NA" as a real value
train = pd.read_csv(
    RAW / "train.csv",
    dtype={"ID": str, "TransactionId": str, "LocationId": str, "StockItemId": str},
    keep_default_na=False,
    na_values=[""],
)
test = pd.read_csv(RAW / "test.csv", dtype={"ID": str})
sample = pd.read_csv(RAW / "SampleSubmission.csv", dtype={"ID": str})
cfg = json.load(open(RAW / "label_config.json"))

print("train :", train.shape)
print("test  :", test.shape)
print("sample:", sample.shape)
print("memory (MB):", round(train.memory_usage(deep=True).sum() / 1e6, 1))
train.head()

In [ ]:
# column types
train.dtypes

## 2. Missing values and the literal NA

In [ ]:
# real blanks per column
train.isna().sum()

In [ ]:
# "NA" is a real category name in this data, check how often it shows up per column
(train == "NA").sum()[lambda s: s > 0]

## 3. Duplicates

In [ ]:
# fully identical rows
n_dup_rows = train.duplicated().sum()

# same basket and same line number should be unique
n_dup_lines = train.duplicated(["TransactionId", "LineItemSequence"]).sum()

# one basket should belong to one customer
cust_per_basket = train.groupby("TransactionId")["ID"].nunique()

print("identical rows          :", n_dup_rows)
print("repeated basket + line  :", n_dup_lines)
print("baskets with >1 customer:", (cust_per_basket > 1).sum())

## 4. Date coverage

In [ ]:
train["ts"] = pd.to_datetime(train["DateTimeZA"], format="mixed")
print("first row:", train["ts"].min())
print("last row :", train["ts"].max())

monthly = train.groupby(train["ts"].dt.to_period("M")).agg(
    rows=("ID", "size"),
    baskets=("TransactionId", "nunique"),
    customers=("ID", "nunique"),
)
monthly

The first month (March 2024) is only partly covered, the data starts on the 19th.
History ends on 30 Nov 2025, and the hidden test window is Dec 2025 to Feb 2026.

## 5. Test customers versus train

In [ ]:
in_train = test["ID"].isin(train["ID"])
print("test IDs found in train:", in_train.mean())
print("test IDs match the sample submission:", set(test["ID"]) == set(sample["ID"]))
print("unique customers in train:", train["ID"].nunique())
print("test customers:", test["ID"].nunique())

In [ ]:
# rule 1 says scored customers have at least 3 baskets before the cutoff
cutoff = pd.Timestamp(cfg["public_history_end_exclusive"])
baskets_before = train[train["ts"] < cutoff].groupby("ID")["TransactionId"].nunique()
eligible = baskets_before[baskets_before >= 3].index

print("eligible at the test cutoff:", len(eligible))
print("test customers that are eligible:", test["ID"].isin(eligible).mean())

In [ ]:
# how long ago did each test customer last visit, this drives Inactivity
last_seen = train.groupby("ID")["ts"].max()
days_since = (cutoff - last_seen.reindex(test["ID"])).dt.days
days_since.describe(percentiles=[0.25, 0.5, 0.75, 0.9])

## 6. Categories versus the label config

In [ ]:
train["LineItemType"].value_counts()

In [ ]:
train["ItemCategoryLevel2"].value_counts()

In [ ]:
# every category in the data should appear in the config mapping
in_data = set(train["ItemCategoryLevel2"].unique())
in_cfg = set(cfg["category_mapping"].keys())
print("in data but not in config:", in_data - in_cfg)
print("in config but not in data:", in_cfg - in_data)

In [ ]:
# fuel rows that are not Sale_Fuel, the rules say these 3 rows stay out of the fuel target
train[(train["ItemCategoryLevel2"] == "Fuel") & (train["LineItemType"] != "Sale_Fuel")]

## 7. Negative values, returns and adjustments

In [ ]:
num_cols = ["TransactionTotalInclAmount", "Quantity", "TotalExclAmount", "TotalModifierAmount", "UnitSellingExclPrice"]
train[num_cols].describe().T

In [ ]:
# negatives are returns or corrections, they reduce the net totals
neg = pd.DataFrame({
    "negative": (train[num_cols] < 0).sum(),
    "zero": (train[num_cols] == 0).sum(),
})
neg

In [ ]:
# adjustment items never count towards targets or activity
train[train["ItemName"].isin(cfg["adjustment_items"])].groupby("ItemName")["TotalExclAmount"].agg(["size", "sum"])

In [ ]:
# the basket total is repeated on every line, so it must be constant inside a basket
total_per_basket = train.groupby("TransactionId")["TransactionTotalInclAmount"].nunique()
print("baskets with more than one total value:", (total_per_basket > 1).sum())

In [ ]:
# fuel price per litre should sit in a narrow band, anything far out is worth a look
fuel = train[train["LineItemType"] == "Sale_Fuel"]
price = fuel["TotalExclAmount"] / fuel["Quantity"].replace(0, np.nan)
price.describe(percentiles=[0.01, 0.5, 0.99])

## 8. Rebuild the official scaling constants

`label_config.json` gives s for each target, calculated over five calibration quarters.
If our label code produces the same numbers, we know our training targets match the ones Zindi scores on.

In [ ]:
import sys
sys.path.insert(0, "../src")

from sasol_clv.pipelines.data_processing.nodes import clean_transactions
from sasol_clv.pipelines.feature_engineering.labels import build_labels

tx = clean_transactions(train.drop(columns=["ts"]), cfg)

logs_fuel, logs_nonfuel = [], []
for c in cfg["calibration_cutoffs"]:
    lab = build_labels(tx, c, cfg)
    logs_fuel.append(np.log1p(lab["y_fuel"].clip(lower=0)))
    logs_nonfuel.append(np.log1p(lab["y_nonfuel"].clip(lower=0)))
    print(c, "customers:", len(lab))

s_fuel = np.std(np.concatenate(logs_fuel))
s_nonfuel = np.std(np.concatenate(logs_nonfuel))
print()
print("our s fuel   :", s_fuel, " official:", cfg["normalization"]["CLV_fuel"])
print("our s nonfuel:", s_nonfuel, " official:", cfg["normalization"]["CLV_nonfuel"])

## Findings

1. No real blanks. The category "NA" is a real value (1 699 rows) and must be read as text, which is why we pass `keep_default_na=False`.
2. Every test customer is in train and has at least 3 baskets before 1 Dec 2025, so we can build features for all of them.
3. All Level2 categories in the data are in the config mapping.
4. There are a few hundred negative rows (returns and corrections) and 317 adjustment rows. Both are handled by the label rules.
5. Our rebuilt scaling constants match the official ones exactly, so our training labels follow the same rules Zindi uses.
6. March 2024 is a partial month. Our earliest training cutoff is June 2024 so every customer has some history.